# 26. Exploratory E2 — was there calm before the storm?
*Exploratory: added after the main results (PREREGISTRATION.md, Amendment A1).*

Danielsson, Valenzuela & Zer (2018) looked at 60 countries over up to 211 years: **long spells of volatility below its usual level** came before banking crises — calm encourages borrowing and risk ('stability is destabilising', Minsky). Was that true before our crises? A long history is needed, so we use **monthly** share prices from the OECD (via FRED): Ireland from 1955, the US from 1957, Germany from 1960.

* **Needs:** the three `FRED_*.csv` files. **Makes:** nothing new.

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

## Ireland, step by step

In [2]:
from pandas import read_csv
prices = read_csv(folder + 'FRED_SPASTT01IEM661N.csv', index_col=0, parse_dates=True)
prices.head()

,SPASTT01IEM661N
observation_date,
1955-01-01,0.768004
1955-02-01,0.776709
1955-03-01,0.780726
1955-04-01,0.782065
1955-05-01,0.777378


In [3]:
prices = prices['SPASTT01IEM661N'].dropna().loc[:'2023-06-30']
from numpy import log, sqrt
r = log(prices).diff().dropna()                         # monthly returns
r = r.clip(r.quantile(0.005), r.quantile(0.995))        # cap the most extreme 0.5% at each end
r.tail()

observation_date
2023-02-01    0.046107
2023-03-01    0.020931
2023-04-01    0.006187
2023-05-01    0.025011
2023-06-01    0.006868
Name: SPASTT01IEM661N, dtype: float64

## Annual volatility, from July to June
As in the paper, a 'year' runs from **July to June** (July 2006 – June 2007 = year 2007). `(r.index.month >= 7)` is `True` (counted as 1) from July on, so those months move to the next year's label.

In [4]:
year_label = r.index.year + (r.index.month >= 7)
groups = r.groupby(year_label)
sigma = groups.std() * sqrt(12)
sigma = sigma[groups.size() == 12]       # complete years only
sigma.tail()

observation_date
2019    0.150643
2020    0.247479
2021    0.101470
2022    0.154715
2023    0.150167
Name: SPASTT01IEM661N, dtype: float64

## The usual level: a one-sided trend
The **Hodrick–Prescott filter** splits a series into a smooth **trend** and the deviations from it (`lamb=5000` sets how smooth). 'One-sided' means: the trend for each year is estimated from the **past years only** — no peeking at the future. So we run the filter again for every year, each time on the data up to that year, and keep its last point.

In [5]:
from statsmodels.tsa.filters.hp_filter import hpfilter
from pandas import Series
trend_values = []
for t in range(9, len(sigma)):                       # start after 10 years of data
    cycle, trend = hpfilter(sigma.values[:t + 1], lamb=5000)
    trend_values.append(trend[-1])
trend = Series(trend_values, index=sigma.index[9:])
trend.tail()

observation_date
2019    0.125915
2020    0.142803
2021    0.135819
2022    0.137718
2023    0.138839
dtype: float64

## Below-trend volatility, and its average over the previous 5 years

In [6]:
below = (sigma - trend).dropna().clip(upper=0)      # 0 when volatility is above its trend
below5 = below.rolling(5).mean().shift(1)           # average of the 5 years BEFORE each year
below5.loc[below5.index.max() + 1] = below.iloc[-5:].mean()
below5 = below5.dropna()
below5.loc[2004:2009].round(3)

observation_date
2004   -0.012
2005   -0.024
2006   -0.021
2007   -0.024
2008   -0.032
2009   -0.032
dtype: float64

For the GFC we look at the five years **before** it started (2003–2007), i.e. the value for 2008. Is it unusually low (more negative) compared with Ireland's whole history? The **percentile** = the share of years with a value as low or lower; **bottom 20% = unusually calm**.

In [7]:
value = below5.loc[2008]
percentile = (below5 <= value).mean() * 100
print('Ireland before the GFC:', round(value, 3), '| percentile:', round(percentile, 1))

Ireland before the GFC: -0.032 | percentile: 40.0


40th percentile: calm, but **not unusually** calm by Ireland's own history.

## All three markets and all three crises
The same steps in a loop.

In [8]:
codes = {'Ireland': 'SPASTT01IEM661N', 'United States': 'SPASTT01USM661N', 'Germany': 'SPASTT01DEM661N'}
crisis_year = {'Global Financial Crisis': 2008, 'Irish sovereign debt crisis': 2010, 'COVID-19': 2020}

for market in codes:
    prices = read_csv(folder + 'FRED_' + codes[market] + '.csv', index_col=0, parse_dates=True)
    prices = prices[codes[market]].dropna().loc[:'2023-06-30']
    r = log(prices).diff().dropna()
    r = r.clip(r.quantile(0.005), r.quantile(0.995))
    groups = r.groupby(r.index.year + (r.index.month >= 7))
    sigma = (groups.std() * sqrt(12))[groups.size() == 12]
    trend_values = []
    for t in range(9, len(sigma)):
        cycle, trend = hpfilter(sigma.values[:t + 1], lamb=5000)
        trend_values.append(trend[-1])
    trend = Series(trend_values, index=sigma.index[9:])
    below = (sigma - trend).dropna().clip(upper=0)
    below5 = below.rolling(5).mean().shift(1)
    below5.loc[below5.index.max() + 1] = below.iloc[-5:].mean()
    below5 = below5.dropna()
    for crisis in crisis_year:
        value = below5.loc[crisis_year[crisis]]
        percentile = (below5 <= value).mean() * 100
        if percentile <= 20:
            verdict = 'UNUSUALLY CALM'
        else:
            verdict = 'not unusual'
        print(market, '|', crisis, '|', round(value, 3), '| percentile', round(percentile, 1), '|', verdict)

Ireland | Global Financial Crisis | -0.032 | percentile 40.0 | not unusual
Ireland | Irish sovereign debt crisis | -0.019 | percentile 61.8 | not unusual
Ireland | COVID-19 | -0.02 | percentile 60.0 | not unusual
United States | Global Financial Crisis | -0.022 | percentile 41.5 | not unusual
United States | Irish sovereign debt crisis | -0.017 | percentile 60.4 | not unusual
United States | COVID-19 | -0.02 | percentile 49.1 | not unusual
Germany | Global Financial Crisis | -0.049 | percentile 4.0 | UNUSUALLY CALM
Germany | Irish sovereign debt crisis | -0.036 | percentile 16.0 | UNUSUALLY CALM
Germany | COVID-19 | -0.027 | percentile 40.0 | not unusual


In [9]:
saved = read_csv(folder + 'results/volatility_paradox_crises.csv')
saved[['market', 'window', 'low5_before', 'percentile', 'unusually_calm']].round(3)

,market,window,low5_before,percentile,unusually_calm
0,Ireland,Global Financial Crisis,-0.032,40.000,False
1,Ireland,Irish sovereign debt crisis,-0.019,61.818,False
2,Ireland,COVID-19,-0.020,60.000,False
3,United States,Global Financial Crisis,-0.022,41.509,False
4,United States,Irish sovereign debt crisis,-0.017,60.377,False
5,United States,COVID-19,-0.020,49.057,False
6,Germany,Global Financial Crisis,-0.049,4.000,True
7,Germany,Irish sovereign debt crisis,-0.036,16.000,True
8,Germany,COVID-19,-0.027,40.000,False


**Result:** before the GFC only **Germany** was unusually calm (4th percentile — its deepest calm in the record). Ireland and the US were calm but not unusually so. No market was unusually calm before COVID-19 — as expected for a shock from outside the financial system.